# Tarea 1 – Parte 2: Del LLM al clasificador

**Pregunta guía:** *¿Pueden convertir titulares peruanos en un clasificador de noticias?*

Se comparan tres formas de clasificar los titulares de `datos/noticias.csv` (La República + El Comercio):

| Enfoque | Qué es | Entrenamiento |
| :--- | :--- | :--- |
| **Zero-shot** | Un modelo de inferencia en lenguaje natural (NLI) decide qué categoría "se deduce" del titular | Ninguno: es el piso a superar |
| **LLM débil** | Gemini etiqueta el dataset completo con un prompt estructurado | Ninguno, pero cuesta llamadas a una API |
| **Fine-tuned** | DistilBERT multilingual ajustado con nuestros titulares | 3 épocas en CPU |

**Decisiones de diseño:**

1. **El split va primero.** Todas las comparaciones se hacen sobre el mismo conjunto de test, que no se usa para
   nada más hasta la evaluación final.
2. **La etiqueta de referencia es la editorial** (`etiqueta`, que sale de la sección donde el diario publicó la
   nota). El LLM es otro "anotador" que se evalúa contra ella.
3. **El clasificador se entrena solo con el titular**, porque en el dashboard el usuario escribe solo un titular:
   el texto de entrenamiento y el de uso tienen que parecerse.

In [ ]:
import gc
import json
import os
import time
from datetime import date
from pathlib import Path
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import seaborn as sns
from dotenv import load_dotenv
from IPython.display import display
from pydantic import BaseModel, Field
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

pio.renderers.default = "plotly_mimetype+png"
pio.renderers["png"].width, pio.renderers["png"].height = 950, 500
pd.set_option("display.max_colwidth", 110)

SEED = 42
CATEGORIAS = ["politica", "economia", "deportes", "ciencia_tecnologia"]
DATOS, MODELO = Path("datos"), Path("modelo")

df = pd.read_csv(DATOS / "noticias.csv")
print("Dataset:", df.shape)
display(pd.crosstab(df["medio"], df["etiqueta"], margins=True, margins_name="Total"))

## 1. Partición 70 / 15 / 15

Split estratificado con `random_state=42`. Se estratifica por **medio + etiqueta** (no solo por etiqueta) para que
train, validación y test tengan la misma mezcla de La República y El Comercio en cada categoría. Así se puede medir
después si el modelo funciona igual de bien con ambos medios.

In [ ]:
estrato = df["medio"] + " | " + df["etiqueta"]
train, temp = train_test_split(df, test_size=0.30, stratify=estrato, random_state=SEED)
val, test = train_test_split(temp, test_size=0.50, stratify=estrato.loc[temp.index], random_state=SEED)

assert set(train["url"]).isdisjoint(val["url"])
assert set(train["url"]).isdisjoint(test["url"])
assert set(val["url"]).isdisjoint(test["url"])

particion = pd.concat([train.assign(particion="train"), val.assign(particion="val"),
                       test.assign(particion="test")])
particion[["url", "particion"]].to_csv(DATOS / "split.csv", index=False)
print({nombre: len(parte) for nombre, parte in [("train", train), ("val", val), ("test", test)]})
display(pd.crosstab(particion["particion"], [particion["medio"], particion["etiqueta"]]))

## 2. Línea base zero-shot (sin fine-tuning)

Se usa `Recognai/bert-base-spanish-wwm-cased-xnli`, un BERT en español entrenado para inferencia en lenguaje natural.
Para cada titular, el modelo evalúa la hipótesis *"Esta noticia trata sobre {categoría}."* con cada una de las 4
categorías y elige la más probable.

* Las categorías se le pasan **en español natural** ("ciencia y tecnología", no `ciencia_tecnologia`): el modelo
  entiende palabras, no códigos.
* Se evalúan **40 titulares del test** (10 por categoría). Es el piso mínimo que los otros enfoques deben superar.

In [ ]:
from transformers import pipeline

ETIQUETA_NATURAL = {"politica": "política", "economia": "economía", "deportes": "deportes",
                    "ciencia_tecnologia": "ciencia y tecnología"}
NATURAL_A_CODIGO = {v: k for k, v in ETIQUETA_NATURAL.items()}

muestra_40 = test.groupby("etiqueta").sample(n=10, random_state=SEED)
zero_shot = pipeline("zero-shot-classification", model="Recognai/bert-base-spanish-wwm-cased-xnli",
                     device="cpu")

inicio = time.perf_counter()
salidas = zero_shot(muestra_40["titular"].tolist(), candidate_labels=list(ETIQUETA_NATURAL.values()),
                    hypothesis_template="Esta noticia trata sobre {}.")
latencia_zs = (time.perf_counter() - inicio) / len(muestra_40)
muestra_40["pred_zs"] = [NATURAL_A_CODIGO[s["labels"][0]] for s in salidas]

acc_zs = accuracy_score(muestra_40["etiqueta"], muestra_40["pred_zs"])
f1_zs = f1_score(muestra_40["etiqueta"], muestra_40["pred_zs"], average="macro")
print(f"ZERO-SHOT (40 titulares del test) | accuracy = {acc_zs:.3f} | F1 macro = {f1_zs:.3f} "
      f"| latencia = {latencia_zs * 1000:.0f} ms por titular")
display(pd.crosstab(muestra_40["etiqueta"], muestra_40["pred_zs"], rownames=["real"], colnames=["zero-shot"]))

del zero_shot, salidas      # se libera el modelo (~440 MB) antes del fine-tuning
gc.collect();

## 3. Supervisión débil con un LLM (Gemini)

El LLM etiqueta **el dataset completo** usando titular y bajada.

* **Prompt estructurado:** rol, definición de cada categoría y reglas para casos borde (ver `INSTRUCCIONES`).
* **Salida tipada:** `response_schema` obliga a Gemini a responder una lista JSON donde `etiqueta_llm` solo puede
  ser una de las 4 categorías (un `Literal`). No hace falta "limpiar" texto libre.
* **Lotes de 20 noticias** por llamada: 20 veces menos llamadas que de a una, menos costo y menos riesgo de
  chocar con el límite de peticiones por minuto del plan gratuito.
* **Robustez:** los alias `gemini-flash-latest` y `gemini-flash-lite-latest` siempre apuntan a la versión vigente
  (los modelos fechados se retiran). Ante un error (503 por saturación, 429 por cuota) se espera y se reintenta, y
  si un modelo falla se pasa al siguiente. La versión exacta que respondió queda en `modelo_llm`.
* **Caché incremental:** cada respuesta se agrega a `datos/raw/llm_respuestas.jsonl`. Si la ejecución se corta,
  se retoma donde quedó, y una vez completo el notebook se re-ejecuta **sin llamar a la API ni necesitar la clave**.
* La API key se lee de `tarea_1/.env` (`GEMINI_API_KEY`), que está en `.gitignore`.

> **Ojo con la "confianza":** es un número que el LLM *dice*, no una probabilidad calculada. En la sección 5 se
> verifica si está calibrada.

In [ ]:
MODELOS_LLM = ["gemini-flash-latest", "gemini-flash-lite-latest"]
TAM_LOTE = 20
PAUSA_ENTRE_LOTES = 4          # segundos: respeta el límite de peticiones por minuto del plan gratuito
RUTA_CACHE_LLM = DATOS / "raw" / "llm_respuestas.jsonl"

INSTRUCCIONES = """Eres editor de un diario peruano. Clasifica cada noticia en UNA de estas 4 categorías:

- politica: Congreso, Poder Ejecutivo, ministros, partidos, elecciones y candidatos, autoridades regionales,
  política internacional, casos judiciales o de corrupción que involucran a políticos o funcionarios.
- economia: mercados, empresas, precios, inflación, tipo de cambio, empleo, impuestos, presupuesto público,
  comercio exterior, minería como actividad económica, AFP, bancos y finanzas personales.
- deportes: competencias, clubes, selecciones, atletas, resultados, fichajes, horarios y transmisiones de partidos.
- ciencia_tecnologia: investigación científica, espacio, salud y medicina como ciencia, medio ambiente y clima,
  tecnología, inteligencia artificial, internet, ciberseguridad, videojuegos.

Reglas:
1. Decide por el TEMA PRINCIPAL de la noticia, no por palabras sueltas.
2. Si una decisión política trata sobre un tema económico, mira qué destaca el titular: el proceso político
   (votación, conflicto entre poderes, autoridades) es politica; el impacto económico (montos, precios,
   afectados) es economia.
3. Usa el titular y la bajada.
4. confianza es tu probabilidad, entre 0 y 1, de que la etiqueta sea correcta.
Responde con un elemento por noticia, usando el mismo id que recibiste."""


class Clasificacion(BaseModel):
    id: int
    etiqueta_llm: Literal["politica", "economia", "deportes", "ciencia_tecnologia"]
    confianza: float = Field(ge=0, le=1)


def pedir_lote(cliente, lote):
    """Clasifica un lote (con columna id) y devuelve una fila por noticia; prueba los modelos en orden."""
    from google.genai import types

    texto = "\n".join(f"[{fila.id}] Titular: {fila.titular}\n     Bajada: {fila.bajada}"
                      for fila in lote.itertuples())
    config = types.GenerateContentConfig(
        system_instruction=INSTRUCCIONES, temperature=0,
        response_mime_type="application/json", response_schema=list[Clasificacion])
    for modelo in MODELOS_LLM:
        for intento in range(3):
            try:
                respuesta = cliente.models.generate_content(
                    model=modelo, contents=f"Clasifica estas {len(lote)} noticias:\n\n{texto}", config=config)
                filas = {c.id: c for c in respuesta.parsed}
                if set(lote["id"]) <= set(filas):
                    return [{**filas[i].model_dump(), "modelo_llm": respuesta.model_version or modelo}
                            for i in lote["id"]]
                print(f"    {modelo}: respuesta incompleta, se reintenta")
            except Exception as e:      # 503 (saturado), 429 (cuota) o una respuesta que no se pudo parsear
                espera = 10 * (intento + 1)
                print(f"    {modelo}: {type(e).__name__} → espera de {espera} s")
                time.sleep(espera)
    raise RuntimeError("Ningún modelo de MODELOS_LLM respondió; revisar la clave o la cuota")


cache_llm = {}
if RUTA_CACHE_LLM.exists():
    for linea in RUTA_CACHE_LLM.read_text(encoding="utf-8").splitlines():
        registro = json.loads(linea)
        cache_llm[registro["url"]] = registro

pendientes = df[~df["url"].isin(cache_llm)]
print(f"En caché: {len(cache_llm)} | por etiquetar: {len(pendientes)}")
if len(pendientes):
    from google import genai

    load_dotenv()
    cliente = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    for inicio in range(0, len(pendientes), TAM_LOTE):
        lote = pendientes.iloc[inicio:inicio + TAM_LOTE].reset_index(drop=True)
        lote["id"] = lote.index + 1
        t0 = time.perf_counter()
        filas = pedir_lote(cliente, lote)
        segundos = time.perf_counter() - t0
        with RUTA_CACHE_LLM.open("a", encoding="utf-8") as archivo:
            for fila, url in zip(filas, lote["url"]):
                registro = {"url": url, "etiqueta_llm": fila["etiqueta_llm"], "confianza": fila["confianza"],
                            "modelo_llm": fila["modelo_llm"], "fecha": date.today().isoformat(),
                            "segundos_por_noticia": segundos / len(lote)}
                cache_llm[url] = registro
                archivo.write(json.dumps(registro, ensure_ascii=False) + "\n")
        print(f"  lote {inicio // TAM_LOTE + 1}: {len(lote)} noticias en {segundos:.1f} s")
        time.sleep(PAUSA_ENTRE_LOTES)

llm = pd.DataFrame(cache_llm.values())
df = df.merge(llm[["url", "etiqueta_llm", "confianza", "modelo_llm", "fecha"]].rename(columns={"fecha": "fecha_llm"}),
              on="url", validate="one_to_one")
assert len(df) == len(llm) and df["etiqueta_llm"].notna().all()

etiquetas_llm = (df[["titular", "etiqueta_llm", "confianza", "modelo_llm", "fecha_llm"]]
                 .rename(columns={"fecha_llm": "fecha"}))
etiquetas_llm.to_csv(DATOS / "etiquetas_llm.csv", index=False, encoding="utf-8")
latencia_llm = llm["segundos_por_noticia"].mean()
print(f"\nGuardado datos/etiquetas_llm.csv: {etiquetas_llm.shape} | modelos que respondieron: "
      f"{df['modelo_llm'].value_counts().to_dict()}")
display(etiquetas_llm.head())

## 4. Auditoría humana a ciegas (20 casos)

Se eligen 20 noticias al azar (`random_state=42`) y se etiquetan **a mano, viendo solo el titular y la bajada**.
La plantilla `datos/auditoria_llm.csv` no muestra la etiqueta del LLM, ni la sección del diario, ni la URL (que
revela la sección): si quien audita ve la respuesta antes de decidir, se ancla a ella y el acuerdo sale inflado.

Recién cuando las 20 etiquetas humanas están completas se comparan con las del LLM. La correspondencia `id → url`
queda en `datos/raw/auditoria_claves.csv`.

In [ ]:
RUTA_AUDITORIA = DATOS / "auditoria_llm.csv"
RUTA_CLAVES = DATOS / "raw" / "auditoria_claves.csv"

if not RUTA_AUDITORIA.exists():
    muestra = df.sample(20, random_state=SEED).reset_index(drop=True)
    muestra["id"] = muestra.index + 1
    muestra[["id", "url"]].to_csv(RUTA_CLAVES, index=False)
    (muestra[["id", "titular", "bajada"]].assign(etiqueta_humana="", comentario="")
     .to_csv(RUTA_AUDITORIA, index=False, encoding="utf-8-sig"))
    raise SystemExit("Plantilla creada en datos/auditoria_llm.csv: completar etiqueta_humana y volver a ejecutar.")

auditoria = (pd.read_csv(RUTA_AUDITORIA, encoding="utf-8-sig").fillna({"comentario": ""})
             .merge(pd.read_csv(RUTA_CLAVES), on="id")
             .merge(df[["url", "etiqueta_llm", "confianza", "etiqueta"]], on="url"))
auditoria["etiqueta_humana"] = auditoria["etiqueta_humana"].str.strip()
assert auditoria["etiqueta_humana"].isin(CATEGORIAS).all(), "Hay etiquetas humanas vacías o mal escritas"

auditoria["acuerdo"] = auditoria["etiqueta_humana"] == auditoria["etiqueta_llm"]
acuerdo_llm_humano = int(auditoria["acuerdo"].sum())
print(f"Acuerdo LLM-humano: {acuerdo_llm_humano}/20 = {acuerdo_llm_humano / 20:.0%}")
display(auditoria[["id", "titular", "etiqueta_humana", "etiqueta_llm", "confianza", "acuerdo", "comentario"]])

## 5. ¿Coinciden el LLM y la etiqueta editorial?

La auditoría mide el acuerdo con una persona en 20 casos. Aquí se mide contra la etiqueta editorial en **todo el
dataset**. Los desacuerdos no siempre son errores del LLM: muchas son noticias ambiguas (una ley del Congreso sobre
las AFP es política y economía a la vez). También se revisa si la confianza que declara el LLM está calibrada.

In [ ]:
acuerdo_editorial = (df["etiqueta_llm"] == df["etiqueta"]).mean()
print(f"Acuerdo LLM vs etiqueta editorial (dataset completo, {len(df)} noticias): {acuerdo_editorial:.1%}")

matriz = pd.crosstab(df["etiqueta"], df["etiqueta_llm"]).reindex(index=CATEGORIAS, columns=CATEGORIAS, fill_value=0)
fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
ax.set(xlabel="Etiqueta del LLM", ylabel="Etiqueta editorial (sección del diario)",
       title="El LLM coincide con la sección del diario en la mayoría de casos")
plt.tight_layout()
plt.show()

df["tramo_confianza"] = pd.cut(df["confianza"], bins=[0, 0.7, 0.8, 0.9, 0.95, 1.0], include_lowest=True)
calibracion = (df.assign(acierto=df["etiqueta_llm"] == df["etiqueta"])
                 .groupby("tramo_confianza", observed=True)
                 .agg(noticias=("acierto", "size"), confianza_media=("confianza", "mean"),
                      acierto_real=("acierto", "mean")).round(3))
display(calibracion)

print("Ejemplos de desacuerdo:")
display(df.loc[df["etiqueta_llm"] != df["etiqueta"], ["medio", "etiqueta", "etiqueta_llm", "confianza", "titular"]]
          .sample(min(10, int((df["etiqueta_llm"] != df["etiqueta"]).sum())), random_state=SEED))

## 6. Fine-tuning de DistilBERT

* **Modelo:** `distilbert-base-multilingual-cased`, una versión destilada de BERT multilingüe: 6 capas en lugar de 12,
  así que entrena e infiere rápido en CPU.
* **Entrada:** solo el titular, truncado a 64 tokens (de sobra para un titular).
* **Configuración:** 3 épocas, batch de 16, `learning_rate=5e-5`, evaluación en validación al final de cada época y
  se conserva la época con mejor **F1 macro** (`load_best_model_at_end`).
* **F1 macro** promedia el F1 de las 4 clases con el mismo peso: una categoría mal clasificada no se esconde detrás
  de las otras.
* **Experimento extra:** el mismo modelo entrenado con las etiquetas del LLM (`etiqueta_llm`) en lugar de las
  editoriales. Responde a la pregunta de la supervisión débil: ¿alcanzan las etiquetas de un LLM para entrenar un
  buen clasificador sin anotar nada a mano?

In [ ]:
import tempfile

from datasets import Dataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer,
                          TrainingArguments, set_seed)

MODELO_BASE = "distilbert/distilbert-base-multilingual-cased"
LABEL2ID = {c: i for i, c in enumerate(CATEGORIAS)}
ID2LABEL = {i: c for c, i in LABEL2ID.items()}
tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)


def a_dataset(parte, columna_etiqueta):
    ds = Dataset.from_dict({"text": parte["titular"].tolist(),
                            "label": parte[columna_etiqueta].map(LABEL2ID).tolist()})
    return ds.map(lambda lote: tokenizer(lote["text"], truncation=True, max_length=64), batched=True)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "f1_macro": f1_score(labels, preds, average="macro")}


def entrenar(columna_etiqueta, carpeta):
    set_seed(SEED)
    modelo = AutoModelForSequenceClassification.from_pretrained(
        MODELO_BASE, num_labels=len(CATEGORIAS), id2label=ID2LABEL, label2id=LABEL2ID)
    args = TrainingArguments(
        output_dir=str(carpeta), num_train_epochs=3, per_device_train_batch_size=16,
        per_device_eval_batch_size=32, learning_rate=5e-5, weight_decay=0.01,
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="f1_macro",
        logging_strategy="epoch", report_to="none", seed=SEED)
    trainer = Trainer(
        model=modelo, args=args, processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
        train_dataset=a_dataset(train, columna_etiqueta), eval_dataset=a_dataset(val, columna_etiqueta),
        compute_metrics=compute_metrics)
    inicio = time.perf_counter()
    trainer.train()
    print(f"Entrenamiento ({columna_etiqueta}) terminado en {time.perf_counter() - inicio:.0f} s")
    return trainer


# El split se hizo antes de unir las etiquetas del LLM: se recuperan por url
train, val, test = (parte[["url"]].merge(df, on="url") for parte in (train, val, test))

trainer = entrenar("etiqueta", MODELO)
historial = pd.DataFrame([h for h in trainer.state.log_history if "eval_f1_macro" in h])
display(historial[["epoch", "eval_loss", "eval_accuracy", "eval_f1_macro"]].round(3))

trainer.save_model(str(MODELO))
tokenizer.save_pretrained(str(MODELO))
(MODELO / "label_map.json").write_text(json.dumps(ID2LABEL, indent=2), encoding="utf-8")
print("Modelo guardado en", MODELO)

In [ ]:
# Experimento extra: mismas condiciones, pero entrenado y validado con las etiquetas del LLM
with tempfile.TemporaryDirectory() as carpeta_tmp:
    trainer_llm = entrenar("etiqueta_llm", carpeta_tmp)
    pred_llm_ft = trainer_llm.predict(a_dataset(test, "etiqueta")).predictions.argmax(-1)
test["pred_ft_llm"] = [ID2LABEL[i] for i in pred_llm_ft]
del trainer_llm
gc.collect();

## 7. Evaluación final en test

Recién aquí se usa el conjunto de test. Las métricas se calculan contra la etiqueta editorial.

In [ ]:
clasificador = pipeline("text-classification", model=trainer.model, tokenizer=tokenizer, device="cpu")
inicio = time.perf_counter()
test["pred_ft"] = [clasificador(t)[0]["label"] for t in test["titular"]]     # de a uno, como en el dashboard
latencia_ft = (time.perf_counter() - inicio) / len(test)

acc_ft = accuracy_score(test["etiqueta"], test["pred_ft"])
f1_ft = f1_score(test["etiqueta"], test["pred_ft"], average="macro")
print(f"FINE-TUNED (test, {len(test)} titulares) | accuracy = {acc_ft:.3f} | F1 macro = {f1_ft:.3f} "
      f"| latencia = {latencia_ft * 1000:.0f} ms por titular\n")
print(classification_report(test["etiqueta"], test["pred_ft"], labels=CATEGORIAS, digits=3))

cm = confusion_matrix(test["etiqueta"], test["pred_ft"], labels=CATEGORIAS)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0],
            xticklabels=CATEGORIAS, yticklabels=CATEGORIAS)
sns.heatmap(cm / cm.sum(axis=1, keepdims=True), annot=True, fmt=".0%", cmap="Blues", cbar=False, ax=axes[1],
            xticklabels=CATEGORIAS, yticklabels=CATEGORIAS)
axes[0].set(title="Matriz de confusión (conteos)", xlabel="Predicción", ylabel="Etiqueta real")
axes[1].set(title="Matriz de confusión (% por fila)", xlabel="Predicción", ylabel="Etiqueta real")
plt.tight_layout()
plt.show()

f1_por_medio = {medio: f1_score(g["etiqueta"], g["pred_ft"], average="macro") for medio, g in test.groupby("medio")}
print("F1 macro por medio:", {m: round(v, 3) for m, v in f1_por_medio.items()})
print("\nErrores del modelo en test:")
display(test.loc[test["pred_ft"] != test["etiqueta"], ["medio", "etiqueta", "pred_ft", "titular"]])

### Reflexión por categoría

_Se redacta después de ver el reporte y los errores de la celda anterior._

## 8. Comparativa y decisión

Para comparar en igualdad de condiciones, además de la métrica de cada enfoque en su propio conjunto, se calcula el
F1 de **todos** sobre los mismos 40 titulares del zero-shot.

In [ ]:
test_40 = test[test["url"].isin(muestra_40["url"])].merge(muestra_40[["url", "pred_zs"]], on="url")


def fila(enfoque, evaluado, y_real, y_pred, pred_40, latencia, costo, internet):
    return {"enfoque": enfoque, "evaluado sobre": evaluado,
            "accuracy": accuracy_score(y_real, y_pred), "F1 macro": f1_score(y_real, y_pred, average="macro"),
            "F1 macro (40 comunes)": f1_score(test_40["etiqueta"], test_40[pred_40], average="macro"),
            "latencia (ms/titular)": round(latencia * 1000), "costo": costo, "internet": internet}


comparativa = pd.DataFrame([
    fila("Zero-shot (BETO-XNLI)", "40 titulares del test", muestra_40["etiqueta"], muestra_40["pred_zs"],
         "pred_zs", latencia_zs, "gratis (CPU local)", "solo para descargar el modelo"),
    fila(f"LLM débil ({df['modelo_llm'].mode()[0]})", f"test ({len(test)})", test["etiqueta"], test["etiqueta_llm"],
         "etiqueta_llm", latencia_llm, "API por llamada (plan gratuito con cuota)", "sí, en cada predicción"),
    fila("DistilBERT fine-tuned (etiqueta editorial)", f"test ({len(test)})", test["etiqueta"], test["pred_ft"],
         "pred_ft", latencia_ft, "gratis (CPU local)", "solo para descargar el modelo"),
    fila("DistilBERT fine-tuned (etiqueta LLM)", f"test ({len(test)})", test["etiqueta"], test["pred_ft_llm"],
         "pred_ft_llm", latencia_ft, "gratis (CPU local) + etiquetado con API", "solo para descargar el modelo"),
]).round(3)
display(comparativa)

metricas = {
    "f1_macro_test": round(f1_ft, 3), "accuracy_test": round(acc_ft, 3), "n_test": len(test),
    "f1_macro_por_medio": {m: round(v, 3) for m, v in f1_por_medio.items()},
    "f1_macro_zero_shot": round(f1_zs, 3), "acuerdo_llm_humano": acuerdo_llm_humano, "n_auditoria": 20,
    "acuerdo_llm_editorial": round(acuerdo_editorial, 3), "modelo_base": MODELO_BASE,
    "modelo_llm": df["modelo_llm"].mode()[0], "fecha_entrenamiento": date.today().isoformat(),
}
(MODELO / "metricas.json").write_text(json.dumps(metricas, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Resultados Test: F1 Macro: {f1_ft:.3f} | Zero-shot: {f1_zs:.3f} | Acuerdo LLM-Humano: {acuerdo_llm_humano}/20")

### Decisión para un entorno real

_Se redacta después de ver la tabla comparativa._